In [1]:
import pandas as pd
import numpy as np

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Load encounters
base_path = r"C:\Users\HP\Desktop\healthcare-operations-predictive-model\data\raw"

encounters = pd.read_csv(base_path + r"\encounters.csv")

# Convert timestamps
encounters["START"] = pd.to_datetime(encounters["START"], utc=True)
encounters["STOP"] = pd.to_datetime(encounters["STOP"], utc=True)

# Calculate target
encounters["DURATION_MINUTES"] = (
    encounters["STOP"] - encounters["START"]
).dt.total_seconds() / 60

# Recreate excluded records
suspect = encounters[
    (encounters["DURATION_MINUTES"] > 43200) &
    (
        encounters["ENCOUNTERCLASS"].isin(
            ["ambulatory", "outpatient"]
        ) |
        encounters["DESCRIPTION"].str.contains(
            "Periodic reevaluation",
            case=False,
            na=False
        )
    )
]

excluded_ids = suspect.index.tolist() + [8980]

model_data = encounters.drop(index=excluded_ids).copy()

print("Modelling records:", len(model_data))

Modelling records: 27877


In [2]:
# Extract date/time features
data = model_data.copy()

data["START_YEAR"] = data["START"].dt.year
data["START_MONTH"] = data["START"].dt.month
data["START_DAY"] = data["START"].dt.day
data["START_HOUR"] = data["START"].dt.hour
data["START_DAYOFWEEK"] = data["START"].dt.dayofweek

# Select features
features = [
    "ENCOUNTERCLASS",
    "CODE",
    "PAYER",
    "START_YEAR",
    "START_MONTH",
    "START_DAY",
    "START_HOUR",
    "START_DAYOFWEEK"
]

X = data[features]
y = data["DURATION_MINUTES"]

# Sort chronologically
data_sorted = data.sort_values("START").copy()

X_sorted = data_sorted[features]
y_sorted = data_sorted["DURATION_MINUTES"]

# Chronological 80/20 split
split_index = int(len(data_sorted) * 0.8)

X_train = X_sorted.iloc[:split_index]
X_test = X_sorted.iloc[split_index:]

y_train = y_sorted.iloc[:split_index]
y_test = y_sorted.iloc[split_index:]

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

Training records: 22301
Testing records: 5576


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor

# Identify categorical and numerical features
categorical_features = [
    "ENCOUNTERCLASS", "CODE", "PAYER"
]

numerical_features = [
    "START_YEAR",
    "START_MONTH",
    "START_DAY",
    "START_HOUR",
    "START_DAYOFWEEK"
]

# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"),
         categorical_features),
        ("num", "passthrough", numerical_features)
    ]
)

# Decision Tree pipeline
decision_tree = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", DecisionTreeRegressor(
        max_depth=10,
        min_samples_leaf=20,
        random_state=42
    ))
])

# Train
decision_tree.fit(X_train, y_train)

# Predict
dt_predictions = decision_tree.predict(X_test)

# Evaluate
dt_mae = mean_absolute_error(y_test, dt_predictions)
dt_rmse = np.sqrt(mean_squared_error(y_test, dt_predictions))
dt_r2 = r2_score(y_test, dt_predictions)

print("Decision Tree Results")
print("MAE:", dt_mae)
print("RMSE:", dt_rmse)
print("R2:", dt_r2)

Decision Tree Results
---------------------
MAE: 60.35989541510101
RMSE: 882.5486867690979
R2: 0.18336004211784196


In [4]:
from sklearn.ensemble import RandomForestRegressor

random_forest = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=100,
        max_depth=12,
        min_samples_leaf=10,
        random_state=42,
        n_jobs=-1
    ))
])

# Train
random_forest.fit(X_train, y_train)

# Predict
rf_predictions = random_forest.predict(X_test)

# Evaluate
rf_mae = mean_absolute_error(y_test, rf_predictions)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_predictions))
rf_r2 = r2_score(y_test, rf_predictions)

print("Random Forest Results")
print("MAE:", rf_mae)
print("RMSE:", rf_rmse)
print("R2:", rf_r2)

Random Forest Results
MAE: 60.92581384989628
RMSE: 881.4239119176993
R2: 0.18544026903635635


In [5]:
# Compare actual and predicted durations
results = pd.DataFrame({
    "Actual_Duration": y_test.values,
    "Predicted_Duration": rf_predictions
})

results["Absolute_Error"] = abs(
    results["Actual_Duration"] - results["Predicted_Duration"]
)

print("Prediction error summary:")
print(results["Absolute_Error"].describe())

print("\nLargest prediction errors:")
print(
    results.sort_values(
        "Absolute_Error",
        ascending=False
    ).head(10)
)

Prediction error summary:
count     5576.000000
mean        60.925814
std        879.394592
min          0.000000
25%          0.007005
50%          0.152875
75%          1.647594
max      34752.188815
Name: Absolute_Error, dtype: float64

Largest prediction errors:
      Actual_Duration  Predicted_Duration  Absolute_Error
733      35340.000000          587.811185    34752.188815
1911     18761.983333         1440.430240    17321.553093
1840     18759.933333         1440.166375    17319.766958
1657     18747.150000         1444.113828    17303.036172
1463     18740.750000         1440.308199    17300.441801
612      17296.516667         1440.090916    15856.425751
1252     17310.416667         1659.052745    15651.363922
2995     15863.533333         1440.065043    14423.468290
1707     14436.233333         1442.383338    12993.849995
768      13350.400000         1759.470015    11590.929985


In [6]:
# Add encounter class to prediction results
results["ENCOUNTERCLASS"] = X_test["ENCOUNTERCLASS"].values

# Error analysis by encounter class
error_by_class = results.groupby("ENCOUNTERCLASS").agg(
    COUNT=("Actual_Duration", "count"),
    MEAN_ACTUAL=("Actual_Duration", "mean"),
    MEDIAN_ACTUAL=("Actual_Duration", "median"),
    MAE=("Absolute_Error", "mean")
).sort_values("MAE", ascending=False)

print(error_by_class)

                COUNT  MEAN_ACTUAL  MEDIAN_ACTUAL          MAE
ENCOUNTERCLASS                                                
inpatient         114  3662.150877         1440.0  2244.782103
emergency         405   147.634568           60.0   112.313953
ambulatory       2324    57.500990           15.0    15.872662
outpatient       1786    15.776932           15.0     0.749836
urgentcare        663    15.000000           15.0     0.150168
wellness          284    15.000000           15.0     0.011238


In [7]:
# Compare duration statistics by encounter class
train_analysis = data_sorted.iloc[:split_index].groupby(
    "ENCOUNTERCLASS"
)["DURATION_MINUTES"].agg(
    ["count", "median", "mean", "max"]
)

test_analysis = data_sorted.iloc[split_index:].groupby(
    "ENCOUNTERCLASS"
)["DURATION_MINUTES"].agg(
    ["count", "median", "mean", "max"]
)

print("TRAINING DURATION DISTRIBUTION")
print(train_analysis)

print("\nTESTING DURATION DISTRIBUTION")
print(test_analysis)

TRAINING DURATION DISTRIBUTION
                count  median         mean           max
ENCOUNTERCLASS                                          
ambulatory      10208    15.0    54.955328    916.016667
emergency        1917    60.0    80.858294  23040.000000
inpatient        1019  1440.0  1487.523602  33120.000000
outpatient       4507    15.0    16.885290    115.166667
urgentcare       3003    15.0    15.000000     15.000000
wellness         1647    15.0    15.000000     15.000000

TESTING DURATION DISTRIBUTION
                count  median         mean           max
ENCOUNTERCLASS                                          
ambulatory       2324    15.0    57.500990    239.000000
emergency         405    60.0   147.634568  35340.000000
inpatient         114  1440.0  3662.150877  18761.983333
outpatient       1786    15.0    15.776932     85.000000
urgentcare        663    15.0    15.000000     15.000000
wellness          284    15.0    15.000000     15.000000


In [8]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.compose import TransformedTargetRegressor

# Dense preprocessing for Gradient Boosting
gb_preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ), categorical_features),
        ("num", "passthrough", numerical_features)
    ]
)

# Gradient Boosting pipeline
gradient_boosting = Pipeline(steps=[
    ("preprocessor", gb_preprocessor),
    ("model", TransformedTargetRegressor(
        regressor=GradientBoostingRegressor(
            n_estimators=150,
            learning_rate=0.05,
            max_depth=3,
            min_samples_leaf=10,
            random_state=42
        ),
        func=np.log1p,
        inverse_func=np.expm1
    ))
])

# Train
gradient_boosting.fit(X_train, y_train)

# Predict
gb_predictions = gradient_boosting.predict(X_test)

# Evaluate
gb_mae = mean_absolute_error(y_test, gb_predictions)
gb_rmse = np.sqrt(mean_squared_error(y_test, gb_predictions))
gb_r2 = r2_score(y_test, gb_predictions)

print("Gradient Boosting Results")
print("MAE:", gb_mae)
print("RMSE:", gb_rmse)
print("R2:", gb_r2)

Gradient Boosting Results
MAE: 64.84243279465551
RMSE: 891.7945222139917
R2: 0.16615970067174402


In [10]:
baseline_predictions = np.full(len(y_test), y_train.median())

baseline_mae = mean_absolute_error(y_test, baseline_predictions)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_predictions))
baseline_r2 = r2_score(y_test, baseline_predictions)

print("Baseline Results")
print("MAE:", baseline_mae)
print("RMSE:", baseline_rmse)
print("R2:", baseline_r2)

Baseline Results
MAE: 102.16142395982781
RMSE: 981.9438280914957
R2: -0.010942767077591009


In [11]:
comparison = pd.DataFrame({
    "Model": [
        "Baseline",
        "Decision Tree",
        "Random Forest",
        "Gradient Boosting"
    ],
    "MAE": [
        baseline_mae,
        dt_mae,
        rf_mae,
        gb_mae
    ],
    "RMSE": [
        baseline_rmse,
        dt_rmse,
        rf_rmse,
        gb_rmse
    ],
    "R2": [
        baseline_r2,
        dt_r2,
        rf_r2,
        gb_r2
    ]
})

comparison

,Model,MAE,RMSE,R2
0,Baseline,102.161424,981.943828,-0.010943
1,Decision Tree,60.359895,882.548687,0.183360
2,Random Forest,60.925814,881.423912,0.185440
3,Gradient Boosting,64.842433,891.794522,0.166160


In [12]:
print("Negative predictions:", (rf_predictions < 0).sum())
print("Minimum prediction:", rf_predictions.min())
print("Maximum prediction:", rf_predictions.max())
print("Mean prediction:", rf_predictions.mean())
print("Mean actual duration:", y_test.mean())

Negative predictions: 0
Minimum prediction: 15.0
Maximum prediction: 1916.4961059617767
Mean prediction: 68.88471850899141
Mean actual duration: 117.16142395982783


In [13]:
error_analysis = pd.DataFrame({
    "Encounter_Class": X_test["ENCOUNTERCLASS"].values,
    "Actual": y_test.values,
    "Predicted": rf_predictions
})

error_analysis["Absolute_Error"] = (
    error_analysis["Actual"] - error_analysis["Predicted"]
).abs()

class_errors = error_analysis.groupby("Encounter_Class").agg(
    Records=("Actual", "count"),
    Mean_Actual=("Actual", "mean"),
    Mean_Predicted=("Predicted", "mean"),
    MAE=("Absolute_Error", "mean")
).round(2)

class_errors

,Records,Mean_Actual,Mean_Predicted,MAE
Encounter_Class,,,,
ambulatory,2324,57.50,56.97,15.87
emergency,405,147.63,87.55,112.31
inpatient,114,3662.15,1524.40,2244.78
outpatient,1786,15.78,15.76,0.75
urgentcare,663,15.00,15.15,0.15
wellness,284,15.00,15.01,0.01


In [14]:
import os
import joblib

project_path = r"C:\Users\HP\Desktop\healthcare-operations-predictive-model"

os.makedirs(os.path.join(project_path, "models"), exist_ok=True)

model_path = os.path.join(
    project_path,
    "models",
    "random_forest_duration_model.joblib"
)

joblib.dump(random_forest, model_path)

print("Model saved successfully:", model_path)

Model saved successfully: C:\Users\HP\Desktop\healthcare-operations-predictive-model\models\random_forest_duration_model.joblib


In [15]:
print(model_data[["ENCOUNTERCLASS", "CODE", "PAYER"]].head())

  ENCOUNTERCLASS       CODE                                 PAYER
0     ambulatory  185347001  b1c428d6-4f07-31e0-90f0-68ffa6ff8c76
1     outpatient  308335008  b1c428d6-4f07-31e0-90f0-68ffa6ff8c76
2     outpatient  185349003  7caa7254-5050-3b5e-9eae-bd5ea30e809c
3       wellness  162673000  b1c428d6-4f07-31e0-90f0-68ffa6ff8c76
4     ambulatory  390906007  42c4fca7-f8a9-3cd1-982a-dd9751bf3e2a
